# 📚 Лабораторная работа 11. RAG — свои документы и знания

Цель: построить полностью локальный учебный Retrieval pipeline без внешнего API.

Мы используем простую TF-IDF-подобную векторизацию, чтобы понять архитектуру. Позже её можно заменить настоящей Embedding Model.


# 1. Импорт

In [ ]:
import math
import re
from collections import Counter

import torch

TOP_K = 3

# 2. Маленькая база документов

In [ ]:
documents = [
    {
        "source": "architecture.md",
        "text": (
            "Основной торговый алгоритм использует Magic Number 112. "
            "Позиции и отложенные ордера Aspid используют Magic Number 115."
        ),
    },
    {
        "source": "license.md",
        "text": (
            "Проверка лицензии выполняется через API. "
            "При сетевой ошибке торговая логика может продолжить работу."
        ),
    },
    {
        "source": "aspid.md",
        "text": (
            "Для работы Aspid требуется hedging-режим торгового счёта. "
            "Проверка режима выполняется один раз при запуске."
        ),
    },
]

for document in documents:
    print(document["source"])
    print(document["text"])
    print()

# 3. Tokenizer

In [ ]:
def tokenize(text):
    return re.findall(
        r"[а-яёa-z0-9]+",
        text.lower(),
    )


print(tokenize(documents[0]["text"]))

# 4. Chunking

In [ ]:
def chunk_words(text, chunk_size=12, overlap=3):
    words = tokenize(text)

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = words[start:end]

        if chunk:
            chunks.append(" ".join(chunk))

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks


print(
    chunk_words(
        documents[0]["text"],
        chunk_size=12,
        overlap=3,
    )
)

# 5. Chunks + Metadata

In [ ]:
chunks = []

for document in documents:
    for index, chunk_text in enumerate(
        chunk_words(
            document["text"],
            chunk_size=12,
            overlap=3,
        )
    ):
        chunks.append(
            {
                "id": f"{document['source']}::{index}",
                "source": document["source"],
                "text": chunk_text,
            }
        )

for chunk in chunks:
    print(chunk)

# 6. Vocabulary

In [ ]:
vocabulary = sorted(
    {
        token
        for chunk in chunks
        for token in tokenize(chunk["text"])
    }
)

token_to_id = {
    token: index
    for index, token in enumerate(vocabulary)
}

print("Vocabulary size:", len(vocabulary))

# 7. Document Frequency

In [ ]:
document_frequency = Counter()

for chunk in chunks:
    for token in set(tokenize(chunk["text"])):
        document_frequency[token] += 1

print(document_frequency.most_common(10))

# 8. TF-IDF-подобный Vector

In [ ]:
def text_to_vector(text):
    tokens = tokenize(text)
    counts = Counter(tokens)

    vector = torch.zeros(
        len(vocabulary),
        dtype=torch.float32,
    )

    total_chunks = len(chunks)

    for token, count in counts.items():
        if token not in token_to_id:
            continue

        tf = count / max(len(tokens), 1)

        df = document_frequency.get(token, 0)

        idf = math.log(
            (total_chunks + 1)
            / (df + 1)
        ) + 1.0

        vector[token_to_id[token]] = tf * idf

    return vector

# 9. Векторизуем Chunks

In [ ]:
chunk_vectors = torch.stack(
    [
        text_to_vector(chunk["text"])
        for chunk in chunks
    ]
)

print("Chunk vectors shape:", chunk_vectors.shape)

# 10. Cosine Similarity

In [ ]:
def cosine_similarity(a, b):
    denominator = (
        torch.linalg.vector_norm(a)
        * torch.linalg.vector_norm(b)
    )

    if denominator.item() == 0:
        return 0.0

    return float(
        torch.dot(a, b) / denominator
    )

# 11. Retrieval

In [ ]:
def retrieve(question, top_k=TOP_K):
    question_vector = text_to_vector(question)

    results = []

    for chunk, vector in zip(
        chunks,
        chunk_vectors,
    ):
        score = cosine_similarity(
            question_vector,
            vector,
        )

        results.append(
            {
                **chunk,
                "score": score,
            }
        )

    results.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    return results[:top_k]

# 12. Запрос про Magic Number

In [ ]:
question = (
    "Какой Magic Number используется "
    "основным алгоритмом?"
)

results = retrieve(question)

for result in results:
    print(
        f"{result['score']:.4f}",
        "|",
        result["source"],
        "|",
        result["text"],
    )

# 13. Запрос про режим счёта

In [ ]:
question = "Какой режим счёта нужен для Aspid?"

for result in retrieve(question):
    print(
        f"{result['score']:.4f}",
        "|",
        result["source"],
        "|",
        result["text"],
    )

# 14. Формируем Context

In [ ]:
question = (
    "Какой Magic Number используется "
    "основным алгоритмом?"
)

retrieved = retrieve(
    question,
    top_k=2,
)

context = "\n\n".join(
    (
        f"[Источник: {item['source']}]\n"
        f"{item['text']}"
    )
    for item in retrieved
)

print(context)

# 15. RAG Prompt

In [ ]:
rag_prompt = f'''
Ответь только на основании КОНТЕКСТА ниже.
Если ответа в контексте нет, скажи:
"В предоставленном контексте ответа нет."

КОНТЕКСТ:
{context}

ВОПРОС:
{question}
'''.strip()

print(rag_prompt)

# 16. Search Tool для будущего агента

In [ ]:
def search_knowledge_base(
    question,
    top_k=3,
):
    return {
        "status": "ok",
        "query": question,
        "results": retrieve(
            question,
            top_k=top_k,
        ),
    }


search_result = search_knowledge_base(
    "Какой Magic Number у Aspid?"
)

print(search_result)

# 17. Функция сборки Prompt

In [ ]:
def build_rag_prompt(question, results):
    context = "\n\n".join(
        (
            f"[Источник: {item['source']}]\n"
            f"{item['text']}"
        )
        for item in results
    )

    return f'''
Ответь только по предоставленному контексту.
Если данных недостаточно, прямо сообщи об этом.

КОНТЕКСТ:
{context}

ВОПРОС:
{question}
'''.strip()


question = "Какой режим счёта нужен Aspid?"

retrieved = retrieve(
    question,
    top_k=2,
)

print(
    build_rag_prompt(
        question,
        retrieved,
    )
)

Полученный Prompt можно передать локальной Qwen через тот же Ollama-клиент, который использовался в главе 9.


# 18. Sources

In [ ]:
for result in retrieved:
    print(
        result["source"],
        "score=",
        round(result["score"], 4),
    )

# 19. 📌 Что нужно запомнить

```text
Documents
↓
Chunks
↓
Vectors
↓
Search

Question
↓
Question Vector
↓
Top-k Chunks
↓
Context
↓
LLM
↓
Answer
```


# 20. 🧩 Эксперименты

Попробуй:

- изменить `chunk_size`;
- изменить `overlap`;
- изменить `TOP_K`;
- добавить новые документы;
- задать вопрос, которого нет в базе;
- посмотреть, какие неправильные Chunks попадают в Top-k.


# 21. ➡️ Следующая глава

# Глава 12. Fine-tuning, LoRA и QLoRA
